In [ ]:
import torch
import numpy as np
from data import make_earth_dataloaders
from model import AmbientGeneratorScoreNet, DPnPScoreWrapper
from plot_earth import plot_earth_train_val_test, plot_earth_latlon, plot_sphere_3d,plot_points_on_earth
from train import train_ism_pathwise
from bel import get_bel
from test_earth import  test_dpnp_sampler_multiple_xtrue,cosine_similarity_vs_steps_for_particle_counts,make_aggregate_plot_from_results,replot_dpnp_results_for_xtrue,make_aggregate_plot_from_results
import matplotlib.pyplot as plt
from utils import get_vmf_f
device = "mps" if torch.backends.mps.is_available() else "cpu"

hidden_dim = 512
n_hidden_layers = 5
MODEL_PATH = "p_score_model_big.pth" 
KAPPA = 30

In [ ]:
def p_score_model(train_loader, val_loader, test_loader, hidden_dim = hidden_dim, n_hidden_layers = n_hidden_layers, T = 1, lr = 2e-4, n_epochs = 30000,n_steps = 100):
    device = "mps" if torch.backends.mps.is_available() else "cpu"
    model = AmbientGeneratorScoreNet(hidden_dim=hidden_dim, n_hidden_layers=n_hidden_layers)
    trained_model = train_ism_pathwise(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        T=T,
        lr=lr,
        n_epochs=n_epochs,
        n_steps=n_steps,
        device=device,
        log_every=100,
        eval_every=1000,
    )
    return trained_model

batch_size = 512
data_name = 'earthquake'
seed = 0
train_loader, val_loader, test_loader, dataset = make_earth_dataloaders(
        data_dir="data",
        name=data_name,   # change to: "fire", "flood", "volcano"
        batch_size=batch_size,
        seed=seed,
        device="cpu",        # keep raw data on CPU; train loop moves batches
    )
plot_earth_train_val_test(train_loader, val_loader, test_loader, title="Earth splits")
    
x_batch, _ = next(iter(test_loader))
plot_earth_latlon(x_batch, title="Testing batch in lat/lon")
plot_sphere_3d(x_batch, title="Testing batch on S^2")
plot_points_on_earth(x_batch)

In [ ]:
trained_p_score_model = p_score_model(train_loader= train_loader, val_loader= val_loader, test_loader= test_loader)
torch.save(trained_p_score_model.state_dict(), MODEL_PATH)


In [ ]:
p_model_loaded2= AmbientGeneratorScoreNet(hidden_dim=hidden_dim, n_hidden_layers=n_hidden_layers)
p_model_loaded2.load_state_dict(torch.load('checkpoint/best.pt',weights_only=False)['model'])
p_model_loaded2.eval().to(device)
p_score2 = DPnPScoreWrapper(p_model_loaded2)

In [ ]:
def aneal_schedule(K:int = 20, K0:int = 5, eta0:float = 0.45, etaK:float = 0.15):
    etas = np.zeros(K)
    for i in range(K):
        if i < K0:
            etas[i] = eta0
        else:
            frac = (i-K0)/(K-K0)
            etas[i] = eta0 * ((etaK/eta0) ** frac)
    return etas

Model_to_load = 'p_score_3_stable.pth'
q_score = get_bel(f_fn= get_vmf_f(kappa=KAPPA))
p_model_loaded = AmbientGeneratorScoreNet(hidden_dim=hidden_dim, n_hidden_layers=n_hidden_layers)
# p_model_loaded.load_state_dict(torch.load(Model_to_load,weights_only=False))
p_model_loaded.eval().to(device)
p_score = DPnPScoreWrapper(p_model_loaded)
eta = aneal_schedule(K=20, K0=5, eta0  = 0.1, etaK  = 0.05)

In [ ]:
num_pairs = 400
summary2 = cosine_similarity_vs_steps_for_particle_counts(
    dataloader=test_loader,
    q_score=q_score,
    p_score=p_score2,
    eta=eta,
    sigma_y=KAPPA,
    particle_counts=[1, 3, 6,10],
    out_samples=11,
    grw_steps=5,
    num_pairs=num_pairs,
    batch_eval_size=16,
    device="mps",
    dtype=torch.float32,
    seed=0,
    show_stderr=True,
    aggregate_mode='mean'
)

summary2 = cosine_similarity_vs_steps_for_particle_counts(
    dataloader=test_loader,
    q_score=q_score,
    p_score=p_score2,
    eta=eta,
    sigma_y=KAPPA,
    particle_counts=[1, 3, 6,10],
    out_samples=11,
    grw_steps=5,
    num_pairs=num_pairs,
    batch_eval_size=16,
    device="mps",
    dtype=torch.float32,
    seed=0,
    show_stderr=True,
    aggregate_mode='min_overall'
)

In [ ]:

results2 = test_dpnp_sampler_multiple_xtrue(
    dataloader=test_loader,
    q_score=q_score,
    p_score=p_score2,
    eta=eta,
    sigma_y=KAPPA,
    num_xtrue=612,
    num_y_per_xtrue=10,
    out_samples=16,
    grw_steps=5,
    device="mps",
    dtype=torch.float32,
    seed=2,
    kappa=25.0,
    plot_results=True
)
save_dict = {
    "results": results2,
    "sigma_y": KAPPA,
    "eta": eta,
    "num_y_per_xtrue": 10,
    "out_samples": 32,
    "grw_steps": 5,
    'num_xtrue': 80,
    'seed': 2,
    'kappa':25
}

torch.save(save_dict, "dpnp_test_pscore2.pt")

In [ ]:
results2_loaded = torch.load('dpnp_test_pscore2.pt', weights_only=False)
n_overlay = 1000
level = 20
make_aggregate_plot_from_results(results=results2_loaded['results'],kappa=25,overlay_count=n_overlay, overlay_points= False, levels=level,mode='x_all',color='white',recon_size=20)

In [ ]:
for i in range(10):
    replot_dpnp_results_for_xtrue(results=results2_loaded['results'],xtrue_idx=3*i)

In [ ]:
results = torch.load('earthquake-300-x.pt', weights_only=False)
n_overlay = 1000
level = 20
make_aggregate_plot_from_results(results=results['results'],kappa=25,overlay_count=n_overlay, levels=level,mode='x_global_mean',color='white',recon_size=10)



In [ ]:
summary = cosine_similarity_vs_steps_for_particle_counts(
    dataloader=test_loader,
    q_score=q_score,
    p_score=p_score,
    eta=eta,
    sigma_y=KAPPA,
    particle_counts=[1, 3, 6],
    out_samples=10,
    grw_steps=5,
    num_pairs=100,
    batch_eval_size=16,
    device="mps",
    dtype=torch.float32,
    seed=0,
    show_stderr=True,
)

In [ ]:
test_dpnp_sampler_multiple_xtrue(
    dataloader=test_loader,
    q_score=q_score,
    p_score=p_score2,
    eta=eta,
    sigma_y=KAPPA,
    num_xtrue=20,
    num_y_per_xtrue=10,
    out_samples=16,
    grw_steps=5,
    device="mps",
    dtype=torch.float32,
    seed=2,
    kappa=25.0,
    plot_results=True
)